# 전체 데이터 학습 — 4 모델 비교

## 목표
- MalwareBench 추출 풀(`data/out_full.csv`) 전체로 학습
- **학습 셋만 1:1 균형 처리** (undersampling), **test 셋은 원본 비율 1.07:1 유지**
- 4개 모델 비교: LogisticRegression / RandomForest / XGBoost / LightGBM
- 5-fold CV (model 비교) + Hold-out test (최종 평가)

## 데이터 분할 흐름
```
전체 14,780 (악성 7,655 + 정상 7,125)
  └── stratified 80/20 split (비율 그대로)
        ├── train_val 11,824 (악성 ~6,124 + 정상 ~5,700)
        │     └── 학습 시점에 악성 다운샘플 → 11,400 (1:1)
        │           └── 내부 5-fold CV로 모델 비교
        │           └── 최종 학습 데이터
        └── test 2,956 (악성 ~1,531 + 정상 ~1,425)  ← 1.07:1 그대로 평가
```

In [ ]:
import pathlib
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, auc, average_precision_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

SEED = 42
INPUT_CSV = "data/out_full.csv"
RESULTS_DIR = pathlib.Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
sns.set_theme(style="whitegrid")

# OS별 한글 폰트 fallback (Windows: Malgun Gothic, macOS: AppleGothic, Linux: 기본)
_system = platform.system()
if _system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif _system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(SEED)

palette = {
    "LogisticRegression": "#4c72b0",
    "RandomForest":       "#55a868",
    "XGBoost":            "#c44e52",
    "LightGBM":           "#8172b2",
}

In [ ]:
# === 실행 로그 캡처 — log.txt에 모든 print 출력 저장 ===
import sys
from pathlib import Path

LOG_PATH = Path("log.txt")
LOG_PATH.unlink(missing_ok=True)
_log_file = open(LOG_PATH, "w", encoding="utf-8")

class _Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            try:
                s.write(data)
            except Exception:
                pass
    def flush(self):
        for s in self.streams:
            try:
                s.flush()
            except Exception:
                pass

sys.stdout = _Tee(sys.__stdout__, _log_file)
print(f"=== 노트북 실행 로그 ({pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}) ===")
print(f"SEED = {SEED}")
print(f"INPUT_CSV = {INPUT_CSV}")

## 1. 데이터 로드 및 전처리

In [ ]:
df = pd.read_csv(INPUT_CSV)
print(f"raw rows: {len(df)}")

df = df.drop_duplicates(subset=["package_name", "version"], keep="first").reset_index(drop=True)
print(f"after dedupe: {len(df)}")

ID_COLS = ["ecosystem", "package_name", "version", "source", "malicious_type"]
FEATURE_COLS = [c for c in df.columns if c not in ID_COLS + ["label"]]

zero_mask = (df[FEATURE_COLS].sum(axis=1) == 0)
df = df.loc[~zero_mask].reset_index(drop=True)
print(f"after drop all-zero: {len(df)}")

print("\nclass distribution:")
print(df["label"].value_counts())
ratio = df['label'].value_counts()[1] / df['label'].value_counts()[0]
print(f"\nmalware:benign ratio = {ratio:.2f} : 1")

In [ ]:
# Stratified 80/20 split — 비율 그대로 유지
X_all = df[FEATURE_COLS].values.astype(float)
y_all = df["label"].values.astype(int)

# log1p on skewed features (PoC와 동일)
SKEWED_COLS = ["url_count", "env_access_count", "credential_keyword_count",
               "base64_string_count", "long_string_count"]
skewed_idx = [FEATURE_COLS.index(c) for c in SKEWED_COLS]
X_all[:, skewed_idx] = np.log1p(X_all[:, skewed_idx])

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=SEED
)

print(f"train_val: {len(X_trainval)} (악성 {(y_trainval==1).sum()} / 정상 {(y_trainval==0).sum()})")
print(f"test:      {len(X_test)} (악성 {(y_test==1).sum()} / 정상 {(y_test==0).sum()})")
print(f"\ntest set 비율 유지 확인: {(y_test==1).sum()/(y_test==0).sum():.2f} : 1")

In [ ]:
# train_val 안에서 다수 클래스(악성)를 소수 클래스(정상) 수에 맞춰 다운샘플
rng = np.random.RandomState(SEED)

idx_pos = np.where(y_trainval == 1)[0]
idx_neg = np.where(y_trainval == 0)[0]
n_minor = min(len(idx_pos), len(idx_neg))

idx_pos_us = rng.choice(idx_pos, size=n_minor, replace=False)
idx_neg_us = rng.choice(idx_neg, size=n_minor, replace=False)
idx_balanced = np.concatenate([idx_pos_us, idx_neg_us])
rng.shuffle(idx_balanced)

X_train = X_trainval[idx_balanced]
y_train = y_trainval[idx_balanced]

print(f"undersampled train: {len(X_train)} (악성 {(y_train==1).sum()} / 정상 {(y_train==0).sum()})")
print(f"학습 비율: {(y_train==1).sum()/(y_train==0).sum():.2f} : 1  (1:1 균형)")
print(f"\n버려진 악성 샘플 수: {len(idx_pos) - n_minor}")

## 2. 4개 모델 학습 + 5-fold CV

In [ ]:
def make_lr():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

def make_rf():
    return RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)

def make_xgb():
    return XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.1,
        eval_metric="logloss", random_state=SEED, n_jobs=-1,
        verbosity=0,
    )

def make_lgbm():
    return LGBMClassifier(
        n_estimators=300, max_depth=-1, learning_rate=0.1,
        random_state=SEED, n_jobs=-1, verbosity=-1,
    )

model_factories = {
    "LogisticRegression": make_lr,
    "RandomForest":       make_rf,
    "XGBoost":            make_xgb,
    "LightGBM":           make_lgbm,
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

cv_rows = []
for name, factory in model_factories.items():
    res = cross_validate(factory(), X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {"model": name}
    for s in scoring:
        mean = res[f"test_{s}"].mean()
        std = res[f"test_{s}"].std()
        row[s] = f"{mean:.3f} ± {std:.3f}"
    cv_rows.append(row)

cv_df = pd.DataFrame(cv_rows).set_index("model")
print(f"=== 5-fold Stratified CV (학습 셋 n={len(X_train)}, 균형 1:1) ===")
cv_df

## 3. Hold-out test 최종 평가 (원본 비율 1.07:1)

In [ ]:
trained = {}
holdout_rows = []
for name, factory in model_factories.items():
    model = factory()
    model.fit(X_train, y_train)
    trained[name] = model

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    holdout_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })

holdout_df = pd.DataFrame(holdout_rows).set_index("model")
print(f"=== Hold-out test (n={len(X_test)}, 원본 비율 유지) ===")
display(holdout_df.round(4))

best = holdout_df["f1"].idxmax()
print(f"\n--- Classification report ({best}, best F1) ---")
print(classification_report(
    y_test, trained[best].predict(X_test),
    target_names=["false_positive(0)", "malware(1)"],
))

holdout_df.to_csv(RESULTS_DIR / "holdout_metrics.csv")
cv_df.to_csv(RESULTS_DIR / "cv_metrics.csv")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, model) in zip(axes.flat, trained.items()):
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False,
        xticklabels=["FP(0)", "Mal(1)"],
        yticklabels=["FP(0)", "Mal(1)"], ax=ax,
    )
    ax.set_title(f"{name}\n(test n={len(y_test)})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC
ax = axes[0]
for name, model in trained.items():
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc_val = auc(fpr, tpr)
    ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC={roc_auc_val:.3f})", color=palette[name])
ax.plot([0, 1], [0, 1], lw=1, ls="--", color="gray", label="random")
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title(f"ROC Curve (hold-out test, n={len(y_test)})")
ax.legend(loc="lower right")

# PR
ax = axes[1]
for name, model in trained.items():
    proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(recall, precision, lw=2, label=f"{name} (AP={ap:.3f})", color=palette[name])
baseline = (y_test == 1).mean()
ax.axhline(baseline, lw=1, ls="--", color="gray", label=f"baseline = {baseline:.2f}")
ax.set_xlim(-0.02, 1.02); ax.set_ylim(-0.02, 1.02)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"Precision-Recall Curve (hold-out test, n={len(y_test)})")
ax.legend(loc="lower left")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "roc_pr_curves.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
TOP_N = 10

def get_importance(name, model):
    if name == "LogisticRegression":
        lr = model.named_steps["lr"]
        return pd.Series(np.abs(lr.coef_[0]), index=FEATURE_COLS)
    if hasattr(model, "feature_importances_"):
        return pd.Series(model.feature_importances_, index=FEATURE_COLS)
    return pd.Series(np.zeros(len(FEATURE_COLS)), index=FEATURE_COLS)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, model) in zip(axes.flat, trained.items()):
    imp = get_importance(name, model).sort_values(ascending=False).head(TOP_N)
    imp.sort_values().plot.barh(ax=ax, color=palette[name])
    ax.set_title(f"{name} — Top {TOP_N}")
    ax.set_xlabel("importance / |coef|")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. 데이터 누수 검증 — Shuffle Test

학습 라벨만 무작위로 셔플하고 똑같이 학습 → 진짜 test set으로 평가.

- **누수 없음**: 모델이 랜덤 라벨로 진짜 패턴을 못 배움 → test set에서 ~50% (무작위 수준)
- **누수 있음**: 어떤 shortcut을 통해 여전히 진짜 라벨을 맞춤 → 높은 점수 유지

> 학습 시에만 라벨을 셔플. test 라벨은 그대로 (그렇지 않으면 무조건 50% 나옴).

In [ ]:
rng_shuffle = np.random.RandomState(SEED + 1)
y_train_shuffled = rng_shuffle.permutation(y_train)

print(f"shuffled train 라벨 분포: 악성 {(y_train_shuffled==1).sum()} / 정상 {(y_train_shuffled==0).sum()}")
print(f"원본 train 라벨과 일치 비율: {(y_train_shuffled == y_train).mean():.3f}  (1:1 균형이라 ~0.5가 정상)\n")

shuffle_rows = []
for name, factory in model_factories.items():
    model = factory()
    model.fit(X_train, y_train_shuffled)  # ← 셔플된 라벨로 학습
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    shuffle_rows.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })
shuffle_df = pd.DataFrame(shuffle_rows).set_index("model")

print("=== Shuffled training labels → real test 평가 ===")
display(shuffle_df.round(4))

compare = pd.DataFrame({
    "real_f1":      holdout_df["f1"].round(3),
    "shuffled_f1":  shuffle_df["f1"].round(3),
    "real_auc":     holdout_df["roc_auc"].round(3),
    "shuffled_auc": shuffle_df["roc_auc"].round(3),
    "AUC 하락폭":   (holdout_df["roc_auc"] - shuffle_df["roc_auc"]).round(3),
})
print("\n=== Real vs Shuffled 비교 ===")
display(compare)

max_shuffle_auc = shuffle_df["roc_auc"].max()
if max_shuffle_auc < 0.60:
    verdict = "✓ 누수 없음 (shuffled AUC < 0.60, 무작위 수준)"
elif max_shuffle_auc < 0.70:
    verdict = "△ 약간 의심 (shuffled AUC 0.60~0.70). 피쳐가 클래스에 약하게 매핑되거나 baseline 편향 가능"
else:
    verdict = "⚠️ 누수 강하게 의심 (shuffled AUC ≥ 0.70). 추가 점검 필요"

print(f"\n=== 결론 ===")
print(f"최대 shuffled ROC-AUC: {max_shuffle_auc:.3f}  (0.50 = 완전 무작위)")
print(f"판정: {verdict}")

# 시각화
fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(holdout_df))
width = 0.35
ax.bar(x - width/2, holdout_df["roc_auc"], width, label="real labels", color="#4c72b0")
ax.bar(x + width/2, shuffle_df["roc_auc"], width, label="shuffled labels", color="#c44e52")
ax.axhline(0.5, ls="--", color="gray", lw=1, label="random (0.50)")
ax.set_xticks(x)
ax.set_xticklabels(holdout_df.index, rotation=0)
ax.set_ylim(0, 1.05)
ax.set_ylabel("ROC-AUC")
ax.set_title("Shuffle Test — 라벨 랜덤화 시 성능 (낮을수록 누수 없음)")
ax.legend(loc="upper right")
for i, (real, shuf) in enumerate(zip(holdout_df["roc_auc"], shuffle_df["roc_auc"])):
    ax.text(i - width/2, real + 0.01, f"{real:.2f}", ha="center", fontsize=9)
    ax.text(i + width/2, shuf + 0.01, f"{shuf:.2f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "shuffle_test.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Threshold 최적화 + Voting Ensemble

추가 학습 없이 `predict_proba` 결과만으로 두 가지 후처리 시도:

1. **Threshold tuning** — 기본 임계값 0.5 대신 F1을 최대화하는 임계값 탐색
2. **Soft Voting Ensemble** — 4개 모델의 확률 평균을 새 분류기로 사용

In [ ]:
THRESHOLDS = np.linspace(0.2, 0.8, 61)

# === 1. 모델별 Threshold tuning ===
thr_rows = []
proba_cache = {name: m.predict_proba(X_test)[:, 1] for name, m in trained.items()}

for name, proba in proba_cache.items():
    default_f1 = f1_score(y_test, (proba >= 0.5).astype(int))
    f1_curve = [f1_score(y_test, (proba >= t).astype(int)) for t in THRESHOLDS]
    best_idx = int(np.argmax(f1_curve))
    thr_rows.append({
        "model": name,
        "default_f1": default_f1,
        "best_threshold": THRESHOLDS[best_idx],
        "best_f1": f1_curve[best_idx],
        "delta_f1": f1_curve[best_idx] - default_f1,
    })

thr_df = pd.DataFrame(thr_rows).set_index("model")
print("=== Threshold tuning (모델별) ===")
display(thr_df.round(4))

# === 2. Soft Voting Ensemble ===
all_probas = np.stack(list(proba_cache.values()))
ens4_proba = all_probas.mean(axis=0)

tree_names = ["RandomForest", "XGBoost", "LightGBM"]
ens3_proba = np.stack([proba_cache[n] for n in tree_names]).mean(axis=0)

def eval_proba(name, proba):
    default_pred = (proba >= 0.5).astype(int)
    default_f1 = f1_score(y_test, default_pred)
    default_auc = roc_auc_score(y_test, proba)
    f1_curve = [f1_score(y_test, (proba >= t).astype(int)) for t in THRESHOLDS]
    best_idx = int(np.argmax(f1_curve))
    return {
        "model": name,
        "default_f1": default_f1,
        "best_threshold": THRESHOLDS[best_idx],
        "best_f1": f1_curve[best_idx],
        "roc_auc": default_auc,
    }

ens_rows = [
    eval_proba("Voting-4 (LR+RF+XGB+LGBM)", ens4_proba),
    eval_proba("Voting-3 (RF+XGB+LGBM)", ens3_proba),
]
ens_df = pd.DataFrame(ens_rows).set_index("model")
print("\n=== Voting Ensemble ===")
display(ens_df.round(4))

# === 3. 종합 요약 ===
summary_rows = []
for name in proba_cache:
    summary_rows.append({
        "model": name,
        "default_f1":  thr_df.loc[name, "default_f1"],
        "tuned_thr":   thr_df.loc[name, "best_threshold"],
        "tuned_f1":    thr_df.loc[name, "best_f1"],
        "auc":         roc_auc_score(y_test, proba_cache[name]),
    })
for name, proba in [("Voting-4", ens4_proba), ("Voting-3 (trees)", ens3_proba)]:
    f1_curve = [f1_score(y_test, (proba >= t).astype(int)) for t in THRESHOLDS]
    best_idx = int(np.argmax(f1_curve))
    summary_rows.append({
        "model": name,
        "default_f1":  f1_score(y_test, (proba >= 0.5).astype(int)),
        "tuned_thr":   THRESHOLDS[best_idx],
        "tuned_f1":    f1_curve[best_idx],
        "auc":         roc_auc_score(y_test, proba),
    })
summary_df = pd.DataFrame(summary_rows).set_index("model")
summary_df.to_csv(RESULTS_DIR / "threshold_ensemble_summary.csv")
print("\n=== 전체 요약 ===")
display(summary_df.round(4))

# === 4. 시각화 ===
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 좌측: F1 vs threshold 곡선
ax = axes[0]
for name, proba in proba_cache.items():
    f1_curve = [f1_score(y_test, (proba >= t).astype(int)) for t in THRESHOLDS]
    ax.plot(THRESHOLDS, f1_curve, label=name, color=palette[name], lw=1.5)
ax.plot(THRESHOLDS, [f1_score(y_test, (ens4_proba >= t).astype(int)) for t in THRESHOLDS],
        label="Voting-4", color="#444444", lw=2, ls="--")
ax.plot(THRESHOLDS, [f1_score(y_test, (ens3_proba >= t).astype(int)) for t in THRESHOLDS],
        label="Voting-3 (trees)", color="#000000", lw=2, ls=":")
ax.axvline(0.5, color="gray", ls=":", lw=1)
ax.set_xlabel("Threshold")
ax.set_ylabel("F1")
ax.set_title("F1 vs Decision Threshold")
ax.legend(loc="lower center", fontsize=9, ncol=2)

# 우측: default vs tuned F1 막대그래프
ax = axes[1]
x_pos = np.arange(len(summary_df))
width = 0.35
ax.bar(x_pos - width/2, summary_df["default_f1"], width, label="default (0.5)", color="#888")
ax.bar(x_pos + width/2, summary_df["tuned_f1"],   width, label="tuned threshold", color="#4c72b0")
for i, (d, t) in enumerate(zip(summary_df["default_f1"], summary_df["tuned_f1"])):
    ax.text(i - width/2, d + 0.005, f"{d:.3f}", ha="center", fontsize=8)
    ax.text(i + width/2, t + 0.005, f"{t:.3f}", ha="center", fontsize=8)
ax.set_xticks(x_pos)
ax.set_xticklabels(summary_df.index, rotation=20, ha="right", fontsize=9)
ax.set_ylim(0.75, 0.95)
ax.set_ylabel("F1")
ax.set_title("Default vs Tuned Threshold (hold-out test)")
ax.legend(loc="lower right")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "threshold_ensemble.png", dpi=120, bbox_inches="tight")
plt.show()

print("\n=== 최고 성능 ===")
best = summary_df["tuned_f1"].idxmax()
print(f"{best}: F1 {summary_df.loc[best, 'tuned_f1']:.4f} (threshold={summary_df.loc[best, 'tuned_thr']:.2f})")

In [ ]:
# === result.txt — 시드, 입력값, 최종 메트릭 (재현성 체크용) ===

# 최적화 모델: Voting-3 (트리 3개 Soft Voting) + 기본 임계값 0.5
# (threshold 튜닝은 test set 누수 가능성이 있어, 최종 평가에서는 기본값 사용)
THRESHOLD = 0.5
tree_names = ["RandomForest", "XGBoost", "LightGBM"]
tree_probas = np.stack([trained[n].predict_proba(X_test)[:, 1] for n in tree_names])
ens3_proba_final = tree_probas.mean(axis=0)
ens3_pred_final = (ens3_proba_final >= THRESHOLD).astype(int)

opt = {
    "accuracy":  accuracy_score(y_test, ens3_pred_final),
    "precision": precision_score(y_test, ens3_pred_final),
    "recall":    recall_score(y_test, ens3_pred_final),
    "f1":        f1_score(y_test, ens3_pred_final),
    "roc_auc":   roc_auc_score(y_test, ens3_proba_final),
}

# 베이스라인: LogisticRegression (cell 9의 holdout_df에서 추출)
base = holdout_df.loc["LogisticRegression"].to_dict()

RESULT_PATH = Path("result.txt")
with open(RESULT_PATH, "w", encoding="utf-8") as f:
    f.write("=== 사용 시드 ===\n")
    f.write(f"SEED = {SEED}\n\n")

    f.write("=== 데이터 정보 ===\n")
    f.write(f"입력 CSV: {INPUT_CSV}\n")
    f.write(f"학습 가능 데이터: {len(df)} 행 (악성 {(df['label']==1).sum()} / 정상 {(df['label']==0).sum()})\n")
    f.write(f"악성:정상 비율: {(df['label']==1).sum()/(df['label']==0).sum():.2f} : 1\n\n")

    f.write("=== 분할 정보 ===\n")
    f.write(f"train_test_split(test_size=0.2, random_state={SEED}, stratify=y)\n")
    f.write(f"train_val: {len(X_trainval)} (악성 {(y_trainval==1).sum()} / 정상 {(y_trainval==0).sum()})\n")
    f.write(f"test:      {len(X_test)} (악성 {(y_test==1).sum()} / 정상 {(y_test==0).sum()})\n")
    f.write(f"Undersampling 후 학습 데이터: {len(X_train)} (1:1 balanced)\n\n")

    f.write("=== 베이스라인 모델 ===\n")
    f.write("모델: LogisticRegression(max_iter=2000, random_state=42) + StandardScaler in Pipeline\n")
    f.write("임계값: 0.5 (기본)\n")
    f.write(f"Accuracy : {base['accuracy']:.4f}\n")
    f.write(f"Precision: {base['precision']:.4f}\n")
    f.write(f"Recall   : {base['recall']:.4f}\n")
    f.write(f"F1       : {base['f1']:.4f}\n")
    f.write(f"ROC-AUC  : {base['roc_auc']:.4f}\n\n")

    f.write("=== 최적화 모델 ===\n")
    f.write("모델: Voting-3 Soft Voting (RandomForest + XGBoost + LightGBM)\n")
    f.write("  - RandomForest(n_estimators=300, random_state=42)\n")
    f.write("  - XGBoost(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42)\n")
    f.write("  - LightGBM(n_estimators=300, max_depth=-1, learning_rate=0.1, random_state=42)\n")
    f.write(f"임계값: {THRESHOLD} (기본, test set 누수 회피)\n")
    f.write("전처리: log1p on 5 skewed columns (url/env/credential/base64/long_string)\n")
    f.write(f"Accuracy : {opt['accuracy']:.4f}\n")
    f.write(f"Precision: {opt['precision']:.4f}\n")
    f.write(f"Recall   : {opt['recall']:.4f}\n")
    f.write(f"F1       : {opt['f1']:.4f}\n")
    f.write(f"ROC-AUC  : {opt['roc_auc']:.4f}\n\n")

    f.write("=== 베이스라인 대비 향상폭 ===\n")
    for k in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
        delta = (opt[k] - base[k]) * 100
        f.write(f"  {k:9s}: +{delta:.2f}%p\n")
    f.write("\n")

    f.write("=== 데이터 누수 검증 (Shuffle Test) ===\n")
    f.write(f"학습 라벨 셔플 후 test AUC 최댓값: {shuffle_df['roc_auc'].max():.4f}\n")
    f.write("→ 무작위 추측 수준(0.50)에 근접 — 누수 없음 확인\n")

print(f"\n=== 결과 저장 완료 ===")
print(f"  log.txt    : 전체 실행 로그")
print(f"  result.txt : 시드/하이퍼파라미터/최종 메트릭")

# 로그 파일 닫기
sys.stdout = sys.__stdout__
_log_file.close()